# 🏋️ Namakan LoRA Fine-Tuning — Training Notebook

**Client:** [CLIENT NAME]

**Model:** [BASE MODEL — e.g. Qwen/Qwen2.5-7B-Instruct]

**Date:** [DATE]

---

## Setup

In [ ]:
# Install dependencies
!pip install -q transformers peft trl accelerate bitsandbytes
!pip install -q datasets scipy wandb

from google.colab import drive
drive.mount('/content/drive')

## 1. Load Data from GCS

Data uploaded via signed URL → GCS bucket

In [ ]:
import os

# GCS bucket (set by trainer)
GCS_BUCKET = "[GCS_BUCKET_NAME]"
TRAIN_DATA = "[TRAIN_DATA_FILE]"  # e.g. data/train.jsonl
VAL_DATA = "[VAL_DATA_FILE]"      # e.g. data/val.jsonl

# Or load directly from Google Drive
# DATA_PATH = "/content/drive/MyDrive/namakan/training_data/"

print(f"✅ Data path: {TRAIN_DATA}")

## 2. Load Base Model + Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "[BASE_MODEL]"  # e.g. "Qwen/Qwen2.5-7B-Instruct"

# Quantization config — 4-bit for T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("🔄 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Model loaded: {MODEL_NAME}")

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load JSONL dataset
train_ds = load_dataset("json", data_files=TRAIN_DATA, split="train")
val_ds = load_dataset("json", data_files=VAL_DATA, split="train")

print(f"📚 Train: {len(train_ds)} examples")
print(f"📚 Val: {len(val_ds)} examples")

In [ ]:
# Tokenize function
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])

print("✅ Tokenized")

## 4. Initialize LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LoRA config — tune these per project
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

## 5. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/namakan/adapters/[CLIENT_NAME]/"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    weight_decay=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    tokenizer=tokenizer,
)

print("🚀 Starting training...")
trainer.train()

In [ ]:
# Save final adapter
trainer.save_model(f"{OUTPUT_DIR}/final")
print(f"✅ Saved to {OUTPUT_DIR}/final")

## 6. Evaluate

In [ ]:
import math

eval_results = trainer.evaluate()
print(f"📊 Eval loss: {eval_results['eval_loss']:.4f}")
print(f"📊 Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

---

## Next Steps

1. Download adapter from Google Drive
2. Merge with base model (optional)
3. Deploy to inference stack
4. Share this notebook with client